# Udział OZE a cena energii

**Pytanie tego notebooka:** czy wysoka generacja z wiatru i fotowoltaiki obniża cenę hurtową,
a jeśli tak, to o ile i od jakiego progu?

Dane: `data/processed/energia_pl.csv`, 76 512 kwadransów z okresu 14.06.2024 - 19.08.2026.
Kluczowa zmienna: `res_share` = (pv + wi) / demand, czyli udział wiatru i fotowoltaiki
w pokryciu zapotrzebowania KSE.

**Zastrzeżenie:** RCE to cena hurtowa na rynku dnia następnego, nie cena z faktury odbiorcy.

## 1. Wczytanie i kontrola danych

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_DIR = Path.cwd().resolve().parent
PROCESSED_CSV = PROJECT_DIR / "data" / "processed" / "energia_pl.csv"

In [3]:
df = pd.read_csv(PROCESSED_CSV)
df.shape

(76512, 17)

In [4]:
print(f"date range: {df['business_date'].min()} - {df['business_date'].max()}")

date range: 2024-06-14 - 2026-08-19


In [5]:
df.head()

,dtime,period,rce_pln,dtime_utc,period_utc,business_date,publication_ts,publication_ts_utc,pv,wi,demand,hour,day_of_week,month,is_weekend,day_name,res_share
0,2024-06-14 00:15:00,00:00 - 00:15,876.10,2024-06-13 22:15:00,22:00 - 22:15,2024-06-14,2024-06-13 17:05:05,2024-06-13 15:05:05.000000,0.0,299.813,16295.296,0,4,6,False,Friday,0.018399
1,2024-06-14 00:30:00,00:15 - 00:30,876.10,2024-06-13 22:30:00,22:15 - 22:30,2024-06-14,2024-06-13 17:05:05,2024-06-13 15:05:05.000000,0.0,320.835,15964.026,0,4,6,False,Friday,0.020097
2,2024-06-14 00:45:00,00:30 - 00:45,876.10,2024-06-13 22:45:00,22:30 - 22:45,2024-06-14,2024-06-13 17:05:05,2024-06-13 15:05:05.000000,0.0,341.506,15772.470,0,4,6,False,Friday,0.021652
3,2024-06-14 01:00:00,00:45 - 01:00,876.10,2024-06-13 23:00:00,22:45 - 23:00,2024-06-14,2024-06-13 17:05:05,2024-06-13 15:05:05.000000,0.0,349.510,15695.535,0,4,6,False,Friday,0.022268
4,2024-06-14 01:15:00,01:00 - 01:15,577.43,2024-06-13 23:15:00,23:00 - 23:15,2024-06-14,2024-06-13 17:05:05,2024-06-13 15:05:05.000000,0.0,362.899,15606.192,1,4,6,False,Friday,0.023254


## 2. Korelacje: co z czym jest powiązane

Liczę **tylko dla zmiennych ciągłych**: cena, generacja PV, generacja wiatrowa, zapotrzebowanie
i udział OZE. Świadomie pomijam `hour`, `month` i `day_of_week`, bo są **cykliczne**: godzina 23
sąsiaduje z godziną 0 w rzeczywistości, ale liczbowo dzielą je 23 jednostki. Korelacja liniowa
zakłada, że wzrost o 1 znaczy to samo na całej skali, a dla tych kolumn to nieprawda.

Obok Pearsona liczę **Spearmana**, czyli korelację na rangach. Pearson mierzy zależność liniową
i jest wrażliwy na wartości skrajne, a ceny sięgają tu od -1458 do +2752 PLN/MWh. Jeśli obie
miary dają podobny wynik, zależność jest solidna. Jeśli się rozjeżdżają, korelację robią
pojedyncze obserwacje odstające.

In [6]:
df[['rce_pln', 'pv', 'wi', 'demand', 'res_share']].corr(method="pearson")

,rce_pln,pv,wi,demand,res_share
rce_pln,1.000000,-0.488371,-0.204053,0.405130,-0.679303
pv,-0.488371,1.000000,-0.266522,0.146418,0.814814
wi,-0.204053,-0.266522,1.000000,-0.016592,0.289202
demand,0.405130,0.146418,-0.016592,1.000000,-0.064559
res_share,-0.679303,0.814814,0.289202,-0.064559,1.000000


In [7]:
df[['rce_pln', 'pv', 'wi', 'demand', 'res_share']].corr(method="spearman")

,rce_pln,pv,wi,demand,res_share
rce_pln,1.000000,-0.353718,-0.193104,0.422094,-0.688598
pv,-0.353718,1.000000,-0.275406,0.266849,0.668938
wi,-0.193104,-0.275406,1.000000,-0.037417,0.352043
demand,0.422094,0.266849,-0.037417,1.000000,-0.037347
res_share,-0.688598,0.668938,0.352043,-0.037347,1.000000


In [8]:
hourly_profiles = df.groupby("hour").agg(
    price=("rce_pln", "median"),
    res_share=("res_share", "median"),
)

profile_corr = hourly_profiles["price"].corr(hourly_profiles["res_share"])
quarter_corr = df["rce_pln"].corr(df["res_share"])

print(f"correlation, hourly profiles (24 points): {profile_corr:.3f}")
print(f"correlation, quarter-hours (76,512):    {quarter_corr:.3f}")

hourly_profiles.round(3)

correlation, hourly profiles (24 points): -0.725
correlation, quarter-hours (76,512):    -0.679


,price,res_share
hour,,
0,448.595,0.154
1,427.340,0.157
2,414.960,0.157
3,413.420,0.156
4,417.220,0.155
5,436.640,0.161
6,512.160,0.177
7,537.835,0.239
8,488.185,0.331


### Co z tych macierzy wynika

**Główna zależność się broni.** Cena wobec udziału OZE: Pearson **-0,679**, Spearman **-0,689**.
Praktycznie identyczne, więc związek jest monotoniczny i nie trzyma się na skrajnościach.

Dla porównania cena wobec samej fotowoltaiki: Pearson -0,488, Spearman -0,354. Tu rozjazd jest
duży, czyli Pearson jest zawyżony przez przypadki bardzo wysokiej produkcji PV zbiegającej się
z bardzo ujemną ceną.

**Druga zmienna zakłócająca wykluczona jedną liczbą.** Zapotrzebowanie wobec udziału OZE:
**-0,065**, czyli praktycznie zero. Można było zarzucić, że wysoki udział OZE wypada w godzinach
niskiego popytu, więc analiza mierzy popyt, a nie podaż. Ta liczba mówi, że nie: udział OZE
jest niemal niezależny od zapotrzebowania.

**Druga strona tego samego mechanizmu.** Zapotrzebowanie wobec ceny: **+0,41**. Wyższy popyt
zmusza system do sięgnięcia po droższe jednostki.

**Ciekawostka o sensie fizycznym.** Fotowoltaika wobec wiatru: **-0,27**, korelacja ujemna.
Wyże baryczne dają bezchmurne niebo i słaby wiatr, niże odwrotnie.

**Pozorna sprzeczność wyjaśniona.** Wiatr wobec ceny to tylko -0,20, a w teście nocnym
(sekcja 5) wysoka generacja wiatrowa obniża medianę ceny o 90,8%. Korelacja liczona na całej
dobie **rozmywa efekt warunkowy**: wiatr działa mocno przy niskim zapotrzebowaniu, a w ciągu
dnia ginie w tle. To ta sama lekcja co w notebooku 01, gdzie globalna średnia równa medianie
ukrywała dwa przeciwstawne ogony.

**Dlaczego dwie różne korelacje ceny z udziałem OZE.** Wersja na 24 uśrednionych godzinach daje
**-0,725**, a na 76 512 kwadransach **-0,679**. Pierwsza jest silniejsza, bo uśrednienie
wygładza szum: znikają dni pochmurne, awarie i anomalie pogodowe, a zostaje sam wzorzec dobowy.
Obie liczby są poprawne, mierzą co innego.

Tabela `hourly_profiles` pokazuje oba profile obok siebie: cena spada z 537 do 300 PLN/MWh,
gdy `res_share` rośnie z 0,239 do 0,514. **To jest zmienna zakłócająca, z którą mierzy się
reszta notebooka.** Jest wystarczająco silna, żeby proste zestawienie ceny z udziałem OZE było
bezwartościowe (pokazywałoby głównie porę dnia), ale wystarczająco słaba, żeby kontrola miała
sens: są południa pochmurne i noce wietrzne, więc zostaje realny zapas zmienności.

## 3. Rozkład res_share

Zanim policzę cokolwiek o cenach, muszę wiedzieć, jak wygląda sama zmienna. Decyle są tu
ważniejsze niż kwartyle, bo interesuje mnie górny ogon: gdzie zaczyna się "dużo OZE".

In [9]:
df['res_share'].quantile([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])

0.0    0.000929
0.1    0.058551
0.2    0.096024
0.3    0.132771
0.4    0.173346
0.5    0.222287
0.6    0.282092
0.7    0.353400
0.8    0.443603
0.9    0.555434
1.0    0.869318
Name: res_share, dtype: float64

## 4. Koszyki udziału OZE

Progi stałe co 10 punktów procentowych, nie kwantyle. Powód: w README pisze się
"przy udziale OZE powyżej 60%", a nie "w dziewiątym decylu". Sprawdziłem, że górne
przedziały nie są puste - najrzadszy ma 1235 obserwacji.

Ostatni koszyk nazwany `>70%`, a nie `70-100%`: granica techniczna to 1,0, ale maksimum
w danych wynosi 0,869, więc etykieta ma opisywać dane, a nie kod.

In [10]:
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0]
labels = ["0-10%", "10-20%", "20-30%", "30-40%", "40-50%", "50-60%", "60-70%", ">70%"]

df['res_share_bin'] = pd.cut(df['res_share'], bins=bins, labels=labels)
df.value_counts('res_share_bin').sort_index()

res_share_bin
0-10%     16164
10-20%    18652
20-30%    13153
30-40%     9698
40-50%     7570
50-60%     6146
60-70%     3894
>70%       1235
Name: count, dtype: int64

### Wersja naiwna - świadomie błędna

Poniższa tabela wygląda spektakularnie: mediana ceny spada z 554 do 1,24 PLN/MWh,
czyli o 99,8%, monotonicznie przez wszystkie osiem koszyków.

**Nie można jej ufać.** Koszyk `>70%` to niemal wyłącznie godziny 11-14, a koszyk `0-10%`
to głównie noc i wieczór. Część tego spadku to po prostu profil dobowy z notebooka 01
ubrany w inną zmienną. Kontrola w sekcji 5.

Warta odnotowania jest zmiana znaku różnicy średnia-mediana: w koszykach środkowych średnia
jest poniżej mediany (ogon cen ujemnych), a w dwóch najwyższych znowu powyżej. Przy medianie
leżącej już przy zerze połowa obserwacji jest ujemna, a średnią ciągną w górę te kwadranse,
w których cena mimo wszystko została wysoka.

In [11]:
price_by_res_bin = df.groupby("res_share_bin", observed=True)["rce_pln"].agg(["median", "mean", "count"])
price_by_res_bin

,median,mean,count
res_share_bin,,,
0-10%,553.920,642.505791,16164
10-20%,514.095,555.271789,18652
20-30%,468.390,483.738403,13153
30-40%,420.090,412.840757,9698
40-50%,352.765,318.097316,7570
50-60%,206.260,189.613309,6146
60-70%,29.070,84.923408,3894
>70%,1.240,6.126526,1235


## 5. Kontrola: to samo wewnątrz stałej pory dnia

Dwa testy. W obu **godzina jest niemal stała**, więc nie może już tłumaczyć różnic w cenie.

| Test | Godziny | Co izoluje |
|---|---|---|
| A | 11-14 | dużo PV, duży rozrzut `res_share` między dniami pochmurnymi a słonecznymi |
| B | 0-4 | **PV nie ma wcale** (mediana 0 MW), więc cały `res_share` pochodzi z wiatru |

Test B jest mocniejszy: wiatr nie ma cyklu dobowego, więc efekt jest oczyszczony
z pory dnia niemal całkowicie.

In [12]:
midday = df[df["hour"].between(11, 14)]

price_by_res_midday = (midday.groupby("res_share_bin", observed=True)["rce_pln"].agg(["median", "mean", "count"]))
price_by_res_midday

,median,mean,count
res_share_bin,,,
0-10%,661.930,735.833617,553
10-20%,545.175,596.200202,1288
20-30%,465.790,477.422654,1183
30-40%,416.170,412.016848,1307
40-50%,345.990,310.668948,2082
50-60%,161.315,156.296501,3064
60-70%,13.680,58.150813,2411
>70%,-0.175,-32.208831,864


In [13]:
night = df[df["hour"].between(0, 4)]

price_by_res_night = (night.groupby("res_share_bin", observed=True)["rce_pln"].agg(["median", "mean", "count"]))
price_by_res_night

,median,mean,count
res_share_bin,,,
0-10%,450.85,463.397661,4754
10-20%,434.20,448.879134,5312
20-30%,399.73,404.188973,3047
30-40%,357.00,333.703046,1763
40-50%,241.93,213.184609,909
50-60%,41.68,101.718828,145
60-70%,332.15,333.721000,10


### Wynik kontroli: efekt nie znika, tylko rośnie

Spadek mediany ceny z koszyka `0-10%` do `50-60%`, ten sam odcinek w każdym teście:

| Test | Spadek |
|---|---|
| naiwny, cała doba | 554 -> 206 = **-62,8%** |
| godziny 11-14 | 662 -> 161 = **-75,6%** |
| godziny 0-4 (sam wiatr) | 451 -> 42 = **-90,8%** |

**Efekt rośnie po kontroli, zamiast maleć.** Mechanizm: w wersji naiwnej koszyk `0-10%`
mieszał tanią noc z drogim wieczorem, więc jego poziom był zaniżony. Po ograniczeniu do
południa zostają w nim wyłącznie **pochmurne południa**, czyli godziny z wysokim
zapotrzebowaniem i bez PV, a te są drogie (662 zamiast 554). Punkt odniesienia poszedł
w górę i różnica się powiększyła.

Innymi słowy: pora dnia tego efektu nie zawyżała, tylko **go maskowała**.

**Zastrzeżenie.** W tabeli nocnej koszyk `60-70%` ma 10 obserwacji i wyłamuje się z trendu
(332 PLN/MWh). Przy takiej liczebności mediana jest przypadkowa. Koszyk `>70%` w nocy nie
występuje wcale: sam wiatr rzadko pokrywa ponad 70% zapotrzebowania.

## 6. Ceny ujemne a udział OZE

Główny wynik tego notebooka: odsetek kwadransów z ceną poniżej zera w każdym koszyku.

Sztuczka w pierwszej linii: `rce_pln < 0` daje kolumnę wartości logicznych, a `True` liczy
się jako 1. Dlatego `sum` daje liczbę kwadransów ujemnych, a `mean` od razu ich udział.

In [14]:
df["is_negative"] = df["rce_pln"] < 0

negative_by_bin = (df
                   .groupby("res_share_bin", observed=True)["is_negative"]
                   .agg(["count", "sum", "mean"])
                   .rename(columns={"count": "quarters",
                                    "sum": "negative",
                                    "mean": "% negative"}))
negative_by_bin["% negative"] = (negative_by_bin["% negative"] * 100).round(1)
negative_by_bin

,quarters,negative,% negative
res_share_bin,,,
0-10%,16164,0,0.0
10-20%,18652,0,0.0
20-30%,13153,6,0.0
30-40%,9698,71,0.7
40-50%,7570,350,4.6
50-60%,6146,1011,16.4
60-70%,3894,1243,31.9
>70%,1235,524,42.4


## 7. Porównanie trzech testów w jednym miejscu

Mediana ceny w koszykach `0-10%` i `50-60%` dla całej doby, dla godzin 11-14 i dla godzin 0-4,
plus procentowa zmiana między nimi. To jest tabela, która idzie do README.

In [15]:
segments = {
    "full day": df,
    "hours 11-14": midday,
    "hours 0-4": night,
}

rows = []
for name, subset in segments.items():
    medians = subset.groupby("res_share_bin", observed=True)["rce_pln"].median()
    low, high = medians["0-10%"], medians["50-60%"]
    rows.append({
        "segment": name,
        "bin 0-10%": round(low, 1),
        "bin 50-60%": round(high, 1),
        "change %": round((high - low) / low * 100, 1),
    })

comparison = pd.DataFrame(rows).set_index("segment")
comparison

,bin 0-10%,bin 50-60%,change %
segment,,,
full day,553.9,206.3,-62.8
hours 11-14,661.9,161.3,-75.6
hours 0-4,450.8,41.7,-90.8


## Wniosek

**Zależność jest realna i przeżywa kontrolę na porę dnia.** Nie jest to profil dobowy
w przebraniu: w godzinach nocnych, gdzie fotowoltaiki nie ma w ogóle, wysoka generacja
wiatrowa obniża medianę ceny o **90,8%**.

**Teza z progiem:** ceny ujemne praktycznie nie występują poniżej 30% udziału OZE
(6 kwadransów na 48 tysięcy). Powyżej 70% dotyczą **42,4% kwadransów**, czyli niemal
co drugiego.

**Mechanizm:** wiatr i fotowoltaika mają koszt zmienny bliski zeru, więc wypierają z rynku
droższe jednostki. Gdy generacja przekracza to, co system jest w stanie zagospodarować,
cena schodzi poniżej zera - wytwórcom bardziej opłaca się dopłacić za odbiór energii niż
zatrzymać i ponownie uruchomić blok.

**Ograniczenia:** koszyk `>70%` liczy 1235 obserwacji na 76 512, a w nocy nie występuje wcale.
`his-wlk-cal` jest publikowany ex post, więc ta zależność jest **wyjaśniająca, nie
prognostyczna** - do prognozowania potrzebne byłoby źródło `pk5l-wp` z prognozami PSE.
Zbiór nie zawiera danych o kosztach paliw ani uprawnień CO2, więc poziom cen poza wpływem
OZE pozostaje niewyjaśniony.